# ALL Grand Final Production Training (45/5)

이 노트북은 `ALL_LOSO.ipynb`의 함수/클래스를 재사용하여,
50명 전체 피험자를 `Train 45 / Val 5`로 분할해 최종 통합 모델을 학습하고 저장합니다.

- 샘플링레이트: `ACC/Slow`
- 기본 정규화: `subject_zscore`
- 기본 손실함수: `focal`
- 저장 경로: `/home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ALL/Save_model/ALL_Grand_Final_Production_Model.pt`

In [ ]:
import argparse
import json
import random
from dataclasses import asdict
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch


def import_symbols_from_notebook(nb_path: Path, symbols: List[str]) -> Dict[str, object]:
    with nb_path.open("r", encoding="utf-8") as f:
        raw_nb = json.load(f)

    code_cells = [cell for cell in raw_nb.get("cells", []) if cell.get("cell_type") == "code"]
    if not code_cells:
        raise RuntimeError(f"No code cells found in {nb_path}")

    first_source = code_cells[0].get("source", [])
    if isinstance(first_source, list):
        first_source = "".join(first_source)

    namespace: Dict[str, object] = {}
    # ALL_LOSO의 핵심 정의는 첫 번째 코드 셀에 있으므로 그대로 실행해 재사용한다.
    exec(first_source, namespace)

    missing = [name for name in symbols if name not in namespace]
    if missing:
        raise RuntimeError(f"Missing symbols from {nb_path.name}: {missing}. Check the first cell of ALL_LOSO.ipynb.")

    return {name: namespace[name] for name in symbols}


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Train final ALL modality production model with 45/5 subject split")
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--norm_mode", type=str, default="subject_zscore", choices=["subject_zscore", "none"])
    parser.add_argument("--loss_name", type=str, default="focal")
    parser.add_argument("--seed", type=int, default=42)
    return parser


def main(cli_args: Optional[List[str]] = None) -> Dict[str, object]:
    parser = build_arg_parser()
    if cli_args is None:
        args, _ = parser.parse_known_args()
    else:
        args = parser.parse_args(cli_args)

    notebook_dir = Path.cwd()
    loso_nb_path = notebook_dir / "ALL_LOSO.ipynb"
    if not loso_nb_path.exists():
        raise FileNotFoundError(f"Cannot find source notebook: {loso_nb_path}")

    symbols = import_symbols_from_notebook(
        loso_nb_path,
        symbols=[
            "HyperParameters",
            "find_dataset_root",
            "discover_complete_subjects",
            "build_split_arrays",
            "make_loader",
            "MultiModal1DCNNTransformer",
            "make_criterion",
            "train_one_epoch",
            "evaluate",
            "FS_ACC",
            "FS_SLOW",
        ],
    )

    HyperParameters = symbols["HyperParameters"]
    find_dataset_root = symbols["find_dataset_root"]
    discover_complete_subjects = symbols["discover_complete_subjects"]
    build_split_arrays = symbols["build_split_arrays"]
    make_loader = symbols["make_loader"]
    ModelClass = symbols["MultiModal1DCNNTransformer"]
    make_criterion = symbols["make_criterion"]
    train_one_epoch = symbols["train_one_epoch"]
    evaluate = symbols["evaluate"]
    FS_ACC_VAL = symbols["FS_ACC"]
    FS_SLOW_VAL = symbols["FS_SLOW"]

    np.random.seed(args.seed)
    random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dataset_root = find_dataset_root()
    all_subjects = discover_complete_subjects(dataset_root)

    if len(all_subjects) < 50:
        raise ValueError(f"Need 50 complete subjects, found {len(all_subjects)}")

    all_subjects = sorted(all_subjects, key=lambda x: int(x[3:]))[:50]
    val_subjects = sorted(random.sample(all_subjects, 5), key=lambda x: int(x[3:]))
    train_subjects = sorted([subject for subject in all_subjects if subject not in val_subjects], key=lambda x: int(x[3:]))

    window_seconds = 60
    stride_seconds = 10

    hp = HyperParameters(
        random_seed=args.seed,
        window_seconds=window_seconds,
        stride_seconds=stride_seconds,
        batch_size=args.batch_size,
        epochs=args.epochs,
        learning_rate=args.lr,
        normalization_mode=args.norm_mode,
        loss_name=args.loss_name,
    )

    print("=== Grand Final ALL Modality Training (45/5) ===")
    print(f"Sampling rate (ACC/Slow): {FS_ACC_VAL}Hz / {FS_SLOW_VAL}Hz")
    print(f"Window/Stride (seconds): {window_seconds}/{stride_seconds}")
    print(f"Normalization: {hp.normalization_mode}")
    print(f"Loss: {hp.loss_name}")
    print(f"Train subjects ({len(train_subjects)}): {train_subjects}")
    print(f"Val subjects ({len(val_subjects)}): {val_subjects}")

    x_tr_acc, x_tr_slow, y_tr = build_split_arrays(dataset_root, train_subjects, hp, is_train=True)
    x_va_acc, x_va_slow, y_va = build_split_arrays(dataset_root, val_subjects, hp, is_train=False)

    train_loader = make_loader(x_tr_acc, x_tr_slow, y_tr, hp.batch_size, shuffle=True)
    val_loader = make_loader(x_va_acc, x_va_slow, y_va, hp.batch_size, shuffle=False)

    model = ModelClass(hp=hp).to(device)
    criterion, loss_cfg = make_criterion(y_tr, hp, device)
    optimizer = torch.optim.Adam(model.parameters(), lr=hp.learning_rate, weight_decay=hp.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4)

    best_val_loss = float("inf")
    best_epoch = -1
    wait = 0
    best_final_state = None
    history: List[Dict[str, float]] = []

    for epoch in range(1, hp.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_metrics["loss"])

        history.append({
            "epoch": float(epoch),
            "train_loss": float(train_loss),
            "val_loss": float(val_metrics["loss"]),
            "val_accuracy": float(val_metrics["accuracy"]),
            "val_f1": float(val_metrics["f1"]),
            "lr": float(optimizer.param_groups[0]["lr"]),
        })

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch
            wait = 0
            best_final_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        else:
            wait += 1
            if wait >= hp.patience:
                print(f"Early stopping at epoch {epoch} (best epoch: {best_epoch})")
                break

    if best_final_state is None:
        raise RuntimeError("Training did not produce a valid checkpoint.")

    save_dir = Path("/home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ALL/Save_model")
    save_dir.mkdir(parents=True, exist_ok=True)
    final_model_path = save_dir / "ALL_Grand_Final_Production_Model.pt"

    payload = {
        "model_state_dict": best_final_state,
        "model_name": "ALL_Grand_Final_Production_Model",
        "created_from": "ALL_LOSO.ipynb",
        "sampling_rate_hz": int(FS_ACC_VAL),
        "sampling_rate_slow_hz": int(FS_SLOW_VAL),
        "window_seconds": int(window_seconds),
        "stride_seconds": int(stride_seconds),
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
        "device_used_for_training": str(device),
        "hyperparameters": asdict(hp),
        "normalization_mode": hp.normalization_mode,
        "loss_config": loss_cfg,
        "subject_split": {"train": train_subjects, "val": val_subjects},
        "history": history,
    }

    torch.save(payload, final_model_path)
    print(f"Saved final production model to: {final_model_path}")

    return payload


final_result = main()
final_result

=== Grand Final ALL Modality Training (45/5) ===
Sampling rate (ACC/Slow): 32Hz / 4Hz
Window/Stride (seconds): 60/10
Normalization: subject_zscore
Loss: focal
Train subjects (45): ['Sub1', 'Sub3', 'Sub4', 'Sub5', 'Sub6', 'Sub7', 'Sub9', 'Sub10', 'Sub11', 'Sub12', 'Sub13', 'Sub14', 'Sub15', 'Sub16', 'Sub17', 'Sub20', 'Sub21', 'Sub22', 'Sub23', 'Sub24', 'Sub26', 'Sub27', 'Sub28', 'Sub29', 'Sub30', 'Sub31', 'Sub32', 'Sub33', 'Sub34', 'Sub35', 'Sub36', 'Sub37', 'Sub38', 'Sub40', 'Sub41', 'Sub42', 'Sub43', 'Sub45', 'Sub46', 'Sub47', 'Sub48', 'Sub49', 'Sub50', 'Sub52', 'Sub53']
Val subjects (5): ['Sub2', 'Sub8', 'Sub18', 'Sub44', 'Sub51']
Early stopping at epoch 9 (best epoch: 1)
Saved final production model to: /home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ALL/Save_model/ALL_Grand_Final_Production_Model.pt


{'model_state_dict': {'acc_branch.0.weight': tensor([[[ 0.1427,  0.1518, -0.0509,  0.1726, -0.0463,  0.0354, -0.1030,
             0.1049,  0.1613],
           [-0.1366,  0.1717,  0.0432,  0.1471,  0.0334,  0.0987, -0.0218,
             0.1559,  0.0346],
           [-0.0859,  0.0553, -0.0832, -0.0162, -0.0734,  0.1345, -0.1450,
            -0.0822, -0.0482]],
  
          [[-0.1239,  0.0123, -0.1973,  0.1648, -0.1708,  0.1457,  0.0254,
            -0.0701,  0.1129],
           [ 0.0388,  0.1643,  0.0275, -0.0562,  0.0545, -0.0413,  0.0891,
             0.1831,  0.1190],
           [-0.0781,  0.1165,  0.0426,  0.0997, -0.1072, -0.1791, -0.0665,
            -0.1384,  0.1640]],
  
          [[ 0.0524,  0.0782,  0.0581, -0.0051,  0.1515, -0.1363,  0.0098,
            -0.1340,  0.0591],
           [-0.0574,  0.0680, -0.0305,  0.1682, -0.1033, -0.1037, -0.1056,
             0.1816,  0.0709],
           [ 0.1848, -0.1587, -0.1944, -0.1527, -0.1323,  0.0779,  0.0674,
             0.1576, -0.10